##### import pandas as pd
import os
from sqlalchemy import create_engine, text
import logging
import time

# ---------------- LOGGING SETUP ----------------
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/ingestion_db.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    filemode="a"
)

# ---------------- DATABASE CONNECTION ----------------
engine = create_engine('sqlite:///inventory.db')


# ---------------- INGEST FUNCTION ----------------
def ingest_db(df, table_name, engine):
    try:
        if df.empty:
            logging.warning(f"⚠️ Skipped empty dataframe for table '{table_name}'")
            return

        df.to_sql(table_name, con=engine, if_exists='append', index=False)
        logging.info(f"✅ Appended {df.shape[0]} rows to '{table_name}'")

    except Exception as e:
        logging.error(f"❌ Error loading table {table_name}: {e}")


# ---------------- CSV PROCESSING (CHUNKS) ----------------
def process_csv(file_path, table_name):
    try:
        for chunk in pd.read_csv(file_path, chunksize=10000, low_memory=False):
            ingest_db(chunk, table_name, engine)
    except Exception:
        # fallback encoding
        for chunk in pd.read_csv(file_path, chunksize=10000, encoding='latin1', low_memory=False):
            ingest_db(chunk, table_name, engine)


# ---------------- EXCEL PROCESSING ----------------
def process_excel(file_path, table_name):
    df = pd.read_excel(file_path)

    if df.empty:
        logging.warning(f"⚠️ Excel file empty: {file_path}")
        return

    ingest_db(df, table_name, engine)


# ---------------- MAIN FUNCTION ----------------
def load_raw_data():
    start = time.time()

    if not os.path.exists('data'):
        logging.error("❌ Data folder not found")
        print("❌ 'data' folder not found")
        return

    # ✅ ONLY VALID FILES
    files = [f for f in os.listdir('data') if f.endswith(('.csv', '.xlsx', '.xls'))]

    print("📂 Files found:", files)

    if not files:
        logging.warning("⚠️ No valid files found in data folder")
        print("❌ No valid files in data folder")
        return

    # ---------------- PROCESS FILES ----------------
    for file in files:
        file_path = os.path.join('data', file)

        try:
            print(f"\n🔄 Processing: {file}")

            table_name = file.split('.')[0].lower().strip()

            # ✅ FIXED DROP TABLE (SQLAlchemy)
            with engine.connect() as conn:
                conn.execute(text(f"DROP TABLE IF EXISTS {table_name}"))

            # 🔹 PROCESS FILE TYPE
            if file.endswith('.csv'):
                process_csv(file_path, table_name)

            elif file.endswith('.xls') or file.endswith('.xlsx'):
                process_excel(file_path, table_name)

            print(f"✅ Finished loading: {file}")

        except Exception as e:
            logging.error(f"❌ Error processing {file}: {e}")
            print(f"❌ Error in {file}: {e}")

    # ---------------- TIME TRACKING ----------------
    end = time.time()
    total_time = (end - start) / 60

    logging.info("--------- Ingestion Complete ---------")
    logging.info(f"Total Time Taken: {total_time:.2f} minutes")

    print(f"\n🎉 Ingestion Completed in {total_time:.2f} minutes")


# ---------------- RUN SCRIPT ----------------
if __name__ == "__main__":
    load_raw_data()